# Convolutional Neural Networks (CNN)

- Name: [Your Name]

- Student ID: [Your Student ID]

## Learning Objectives

This assignment uses **prompt programming** with Python and the PyTorch framework to learn core concepts of Convolutional Neural Networks (CNN) and complete image classification on the CIFAR10 dataset. Through this assignment, you will:

1. Understand the principles of image convolution and the role of boundary padding
2. Master the effects and differences of common convolution kernels (mean, Gaussian, sharpening)
3. Learn about image noise types and denoising methods
4. Master basic edge detection methods (Sobel operator, Gaussian first-order derivative)
5. Build a CNN using PyTorch to complete CIFAR10 classification
6. Understand the roles of pooling operations and data augmentation

### Instructions

Each code cell is preceded by a **prompt**. Please enter the prompt into an AI programming tool (such as ChatGPT, Claude, etc.), paste the generated code into the corresponding cell, and run it.

---


## 1. Environment Setup


**Prompt**

Please help me configure the Python environment with the following requirements:

**1. Import the following libraries:**
- Basic libraries: `numpy` (as np), `matplotlib.pyplot` (as plt), `matplotlib` (as mpl), `time`, `warnings`
- Image processing: `convolve2d` from `scipy.signal`, `cv2` (OpenCV), `camera` from `skimage.data`
- Machine learning metrics: `confusion_matrix`, `classification_report`, `accuracy_score` from `sklearn.metrics`
- Visualization: `seaborn` (as sns)
- Deep learning framework: `torch`, `torch.nn`, `torch.optim`, `torchvision.transforms`, `torch.utils.data.DataLoader`, `torchvision.datasets.CIFAR10`
- Use `warnings.filterwarnings('ignore')` to suppress unnecessary warnings

**2. Set matplotlib font parameters:**
- Set `axes.unicode_minus` to `False` to fix minus sign display issues


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import time
import warnings
warnings.filterwarnings('ignore')

# Image processing
from scipy.signal import convolve2d
import cv2
from skimage.data import camera

# Machine learning
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns

# PyTorch deep learning framework
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10

# Fix minus sign display
plt.rcParams['axes.unicode_minus'] = False


## 2. Image Convolution

Convolution is the core operation of CNNs. This section helps you intuitively understand the convolution computation process through manual 2D convolution implementation.


### 2.1 Loading Test Image and Convolution Function


**Prompt**

Please help me implement image loading and a 2D convolution function with the following requirements:

**1. Load the test image:**
- Use `skimage.data.camera()` to load the classic camera test image (a 512x512 grayscale image)
- Convert the image to `np.float64` type for subsequent calculations
- Print the image size in the format: `"Image size: {img.shape}"`

**2. Define a convolution function `conv2d(image, kernel, mode='same')`:**
- Use `scipy.signal.convolve2d` to implement 2D convolution
- The `mode` parameter supports 'same' (output same size as input, automatic padding) and 'valid' (no padding, output shrinks)
- Use `np.clip` to clip output values to [0, 255] range and convert to `np.uint8` type

**3. Define a mean filter kernel (also called box filter):**
- Size 3x3, all elements equal to 1/9
- This kernel averages each pixel with its 3x3 neighborhood, producing a blur effect

**4. Display a comparison of the original and convolved images:**
- Create a canvas with `figsize=(12, 5)`
- Subplot 1: original image, title 'Original Image'
- Subplot 2: convolved image, title '3x3 Mean Filter Result'
- Both use `plt.imshow(..., cmap='gray')` to display in grayscale
- Use `plt.tight_layout()` to adjust layout


**Think:** What changes occurred in the convolved image? Why do edges and details become blurred?


In [ ]:
# TODO: add your code here

### 2.2 Effect of Boundary Padding


**Prompt**

Please help me demonstrate the effect of boundary padding (padding) on convolution results with the following requirements:

**1. Perform convolution with `mode='valid'` (no boundary padding):**
- Use the 3x3 mean kernel `box_kernel` to convolve the image with `mode='valid'`
- Print the output size and compare with the original image size

**2. Create a 2x2 subplot canvas (`figsize=(12, 10)`):**
- Subplot (1,1): original image, title 'Original Image (512x512)'
- Subplot (1,2): valid mode result, title 'Valid Mode (No Padding)'
- Subplot (2,1): same mode result, title 'Same Mode (Zero Padding)'
- Subplot (2,2): difference map between original and same mode result (computed as `img.astype(np.uint8) - conv2d(img, box_kernel, 'same')`), title 'Difference Before/After Convolution'

**3. Use `plt.tight_layout()` to adjust layout**


**Think:** Compare the output image sizes of valid and same modes. Do you now understand why same mode (also called SAME padding) is commonly used in practice? What is the role of boundary padding?


In [ ]:
# TODO: add your code here

### 2.3 Common Convolution Kernel Effects (Mean Filter, Sharpening, Gaussian Filter)


**Prompt**

Please help me demonstrate the effects of different convolution kernels with the following requirements:

**1. Define the following kernels:**
- **Mean filter kernel (5x5)**: `box_kernel_5 = np.ones((5, 5)) / 25.0`, effect: image blur
- **Gaussian kernel (5x5, sigma=1.0)**: Manually generate the Gaussian kernel (without using library functions), formula: `G(x,y) = (1/(2*pi*sigma^2)) * exp(-(x^2+y^2)/(2*sigma^2))`, then normalize so all elements sum to 1. You can:
  - Create a 5x5 grid centered at (2,2) with coordinate range [-2, 2]
  - Loop through each position to compute Gaussian values
- **Sharpening kernel**: `sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])`, effect: enhance image edges and details

**2. Create a 2x2 subplot canvas (`figsize=(12, 10)`):**
- Subplot 1: original image, title 'Original Image'
- Subplot 2: 5x5 mean filter result, title '5x5 Mean Filter (Blur)'
- Subplot 3: Gaussian filter result, title '5x5 Gaussian Filter (sigma=1.0)'
- Subplot 4: sharpening result, title 'Sharpening'
- All use `cmap='gray'`, `plt.axis('off')`
- Use `plt.tight_layout()`

**3. Print the Gaussian kernel:** Use `print(f"Gaussian kernel:\n{np.round(gaussian_kernel, 4)}")` to display the generated kernel matrix


**Think:** Observe the differences between mean filter and Gaussian filter results — does Gaussian filter preserve more edge information while smoothing? How does the sharpening kernel achieve its effect?


In [ ]:
# TODO: add your code here

## 3. Image Noise and Denoising

Images often introduce noise during acquisition and transmission. This section will learn the application of convolution operations in denoising.


### 3.1 Gaussian Noise and Gaussian Filter Denoising


**Prompt**

Please help me demonstrate adding Gaussian noise and denoising with Gaussian filters with the following requirements:

**1. Add Gaussian noise:**
- Generate random noise of the same size as the image: `noise = np.random.normal(0, 25, img.shape)` (mean 0, standard deviation 25)
- Noisy image: `noisy_img = np.clip(img + noise, 0, 255).astype(np.uint8)`

**2. Denoise using Gaussian filters with different sigma values:**
- Generate Gaussian kernels with different sigma values (using the previously defined `gaussian_kernel` function): sigma = [1.0, 2.0, 3.0], kernel size should adjust with sigma (rule of thumb: half-window = 3*sigma, kernel size = 2*3*sigma + 1)
- `kernel_size = 2 * int(3 * sigma) + 1`
- Apply convolution filtering to the noisy image separately

**3. Display comparison (`figsize=(15, 10)`):**
- 2 rows, 3 columns, 6 subplots total:
  - Subplot (1,1): original image (`img.astype(np.uint8)`), title 'Original Image'
  - Subplot (1,2): noisy image, title 'Gaussian Noise (sigma=25)'
  - Subplot (2,1): sigma=1.0 denoising result, title 'Gaussian Filter sigma=1.0'
  - Subplot (2,2): sigma=2.0 denoising result, title 'Gaussian Filter sigma=2.0'
  - Subplot (2,3): sigma=3.0 denoising result, title 'Gaussian Filter sigma=3.0'
- Note: position (1,3) is blank, use `plt.subplot(2, 3, 3); plt.axis('off')` to hide
- All use `cmap='gray'`, `plt.axis('off')`


**Think:** Observe the denoising effects with different sigma values — the larger the sigma, the more thoroughly noise is removed, but the blurrier the image becomes. This is the trade-off between denoising strength and detail preservation.


In [ ]:
# TODO: add your code here

### 3.2 Salt & Pepper Noise and Median Filter Denoising


**Prompt**

Please help me demonstrate adding salt & pepper noise and denoising with median filters with the following requirements:

**1. Add salt & pepper noise:**
- Create a noise mask using `np.random.random(img.shape)` to generate random numbers in [0,1)
- Set noise density `salt_pepper_ratio = 0.05` (about 5% of pixels are corrupted)
- Set pixels with random numbers less than `salt_pepper_ratio/2` to 0 (black, pepper noise)
- Set pixels with random numbers greater than `1 - salt_pepper_ratio/2` to 255 (white, salt noise)
- Assign the noisy image to variable `sp_img`

**2. Denoise using OpenCV median filter:**
- Use `cv2.medianBlur(sp_img, 3)` for 3x3 neighborhood median filtering
- Assign to variable `median_denoised`

**3. Compare Gaussian filter and median filter for salt & pepper noise removal:**
- Generate a sigma=1.0 Gaussian kernel (using `gaussian_kernel` function) and apply Gaussian filtering to `sp_img`
- Assign to variable `gauss_denoised`

**4. Display comparison (`figsize=(12, 8)`):**
- 2 rows, 3 columns (`plt.subplot(2, 3, ...)`):
  - (1,1): original image
  - (1,2): salt & pepper noise image
  - (1,3): median filter denoising result, title 'Median Filter (3x3)'
  - (2,1): Gaussian filter denoising result, title 'Gaussian Filter (sigma=1.0)'
  - (2,2) and (2,3): blank (turn off axes)
- All use `cmap='gray'`, `plt.axis('off')`


**Think:** Which performs better on salt & pepper noise — median filter or Gaussian filter? Why? (Hint: consider the difference between salt & pepper noise pixel values and normal neighborhood pixel values.)
In CNNs, the pooling layer (especially max pooling) has a similar 'taking neighborhood statistics' idea as the median filter.


In [ ]:
# TODO: add your code here

## 4. Edge Detection

Edges are locations in an image where brightness changes significantly. Edge detection can be achieved by computing image gradients, and gradient computation is essentially a convolution operation.


### 4.1 Sobel Operator and Gaussian First-Order Derivative Edge Detection


**Prompt**

Please help me implement Sobel operator edge detection and Gaussian first-order derivative edge detection with the following requirements:

**1. Define Sobel operators (horizontal and vertical directions):**
- `sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])` detects vertical edges (horizontal gradient)
- `sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])` detects horizontal edges (vertical gradient)

**2. Compute gradient magnitude:**
- Use `conv2d` function to compute x and y direction gradients: `gx`, `gy`
- Gradient magnitude: `magnitude = np.sqrt(gx.astype(float)**2 + gy.astype(float)**2)`
- Normalize to [0,255]: scale magnitude to 0-255 range

**3. Gaussian first-order derivative method:**
- First generate Gaussian kernel `gauss_k = gaussian_kernel(5, 1.0)`
- Smooth the image first: `smoothed = conv2d(img.astype(np.uint8), gauss_k)`
- Compute Sobel gradients on the smoothed image
- Get gradient magnitude `gauss_magnitude`

**4. Display comparison (`figsize=(14, 10)`):**
- 2 rows, 3 columns:
  - (1,1): original image
  - (1,2): Sobel X direction gradient, title 'Sobel X (Vertical Edges)'
  - (1,3): Sobel Y direction gradient, title 'Sobel Y (Horizontal Edges)'
  - (2,1): Sobel gradient magnitude, title 'Sobel Edge Detection'
  - (2,2): Gaussian first-order derivative gradient magnitude, title 'Gaussian First-Order Derivative Edge Detection'
  - (2,3): blank
- All use `cmap='gray'`, `plt.axis('off')`


**Think:** The Sobel operator directly performs differentiation on the original image, while the Gaussian first-order derivative smooths first and then differentiates. What are the differences between these two methods?
In CNNs, the first-layer convolution kernels can usually learn patterns similar to Gabor filters or edge detectors, which is similar to the Sobel operator's function, but CNN convolution kernels are learned automatically through data-driven training rather than being hand-designed.


In [ ]:
# TODO: add your code here

## 5. Convolutional Neural Network (CNN)

From this section onward, we will use the PyTorch deep learning framework to build a CNN and complete image classification on the CIFAR10 dataset.
Unlike the hand-designed convolution kernels in the previous sections, CNN convolution kernels will be learned automatically through data-driven training.


### 5.1 CIFAR10 Dataset Loading and Visualization


**Prompt**

Load the CIFAR10 dataset using PyTorch and visualize it with the following requirements:

**1. Set global parameters:**
- Batch size: `batch_size = 64`
- Training epochs: `num_epochs = 15`
- Class names: `classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')`

**2. Define data augmentation preprocessing (unlike the fully-connected network assignment, CNN can use stronger data augmentation):**
- Use `transforms.Compose` to combine the following transformations:
  - `transforms.RandomHorizontalFlip(p=0.5)`: random horizontal flip with probability 0.5
  - `transforms.RandomCrop(32, padding=4)`: random crop, first pad 4 pixels then crop back to 32x32
  - `transforms.ToTensor()`: convert image to tensor
  - `transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])`: normalize pixel values to [-1, 1]

**3. Load the dataset:**
- Training set: `CIFAR10(root='./data', train=True, transform=train_transform, download=True)`
- Test set: `CIFAR10(root='./data', train=False, transform=test_transform, download=True)` (test set uses simple transform, only ToTensor and Normalize, no random augmentation)
- Use `DataLoader` to create data loaders, training set `shuffle=True`, test set `shuffle=False`, `num_workers=2`

**4. Implement image display function `imshow(img)`:**
- Parameter `img` is a torch.Tensor with shape (C, H, W)
- Unnormalize: `img = img / 2 + 0.5`
- Convert to numpy array and adjust dimension order from (C,H,W) to (H,W,C): `np.transpose(npimg, (1, 2, 0))`
- Use `plt.imshow` to display and turn off axes

**5. Visualize sample images:**
- Get the first batch from the training set data loader
- Use `plt.subplots(2, 2)` to create 2x2 subplots
- Display 4 images, each with title as the corresponding class name (use `classes[labels[i]]` to get)
- Use `plt.tight_layout()` to adjust layout

**6. Print dataset information:**
- Training set size: `len(trainset)`
- Test set size: `len(testset)`
- Image dimensions: `trainset[0][0].shape`


In [ ]:
# TODO: add your code here

### 5.2 Random Seeds and Computing Device


**Prompt**

Set random seeds and check the computing device using PyTorch with the following requirements:
1. Use `torch.manual_seed(42)` and `np.random.seed(42)` to set random seeds for reproducible results
2. Use `torch.device('cuda' if torch.cuda.is_available() else 'cpu')` to get the current computing device (CPU or GPU)
3. Print device information in the format: `"Using device: {device}"`


In [ ]:
# TODO: add your code here

### 5.3 CNN Network Structure Definition


**Prompt**

Create a convolutional neural network class using PyTorch with the following requirements:

**1. Class definition:**
- Class name: `SimpleCNN`, inherits from `nn.Module`

**2. Network structure:**
- **Conv layer 1**: `Conv2d(3, 32, 3, padding=1)` -> `BatchNorm2d(32)` -> `ReLU` -> `MaxPool2d(2, 2)`
- **Conv layer 2**: `Conv2d(32, 64, 3, padding=1)` -> `BatchNorm2d(64)` -> `ReLU` -> `MaxPool2d(2, 2)`
- **Conv layer 3**: `Conv2d(64, 128, 3, padding=1)` -> `BatchNorm2d(128)` -> `ReLU` -> `MaxPool2d(2, 2)`
- **Flatten layer**: Use `nn.Flatten()` to flatten feature maps into a 1D vector
- **FC layer 1**: `Linear(2048, 256)` -> `BatchNorm1d(256)` -> `ReLU`
  - Why input is 2048? After 3 pooling operations, 32x32 image size becomes 4x4, multiplied by the previous layer output channels 128: 4x4x128=2048
- **Output layer**: `Linear(256, 10)` (Note: no Softmax needed, because CrossEntropyLoss handles it internally)

**3. Weight initialization (at the end of `__init__`):**
- Define a `_init_weights` method that takes a module `m` as input
- For `nn.Linear` and `nn.Conv2d` layers, use `nn.init.kaiming_normal_(m.weight)` to initialize weights
- For biases, initialize to 0 using `nn.init.constant_(m.bias, 0)`
- Apply the initialization using `self.apply(self._init_weights)`
- Note: BatchNorm layers do not need manual initialization (PyTorch defaults are appropriate)

**4. Forward propagation `forward(self, x)`:**
- Pass sequentially through conv layers, pooling layers -> flatten -> FC layers -> output
- Return output (without Softmax)


**Think:**
- Why is Softmax not added at the output layer here? How is this different from the fully-connected network assignment?
- Why use 3x3 kernels with padding=1? What are the benefits of this design?
- What is the role of pooling layers? How does the feature map size change after each pooling?


In [ ]:
# TODO: add your code here

### 5.4 Model Instantiation and Structure Printing


**Prompt**

Instantiate the SimpleCNN model and print its structure with the following requirements:
1. Create a `SimpleCNN()` instance and move it to the computing device using `.to(device)`
2. Use `print(model)` to print model structure information
3. Iterate through model parameters (`model.parameters()`), count and print total parameters:
   - Use list comprehension: `sum(p.numel() for p in model.parameters())`
   - Format: `f"Total parameters: {total_params:,}"`


In [ ]:
# TODO: add your code here

### 5.5 Loss Function and Optimizer


**Prompt**

Define the loss function and optimizer required for model training with the following requirements:
1. Loss function: Use `nn.CrossEntropyLoss()` (cross-entropy loss), assign to variable `criterion`
2. Optimizer: Use `optim.Adam`, set learning rate to `0.001`, parameters as `model.parameters()`, assign to variable `optimizer`

**Note:** `CrossEntropyLoss` internally includes the Softmax operation, so it receives raw logits rather than probability values after Softmax.
This is why we do not add Softmax at the last layer of the network.
Recall the Softmax + CrossEntropyLoss issue in the fully-connected network assignment — think about: how does this affect training?


In [ ]:
# TODO: add your code here

### 5.6 Training Function


**Prompt**

Create a model training function using PyTorch with the following requirements:

**1. Function definition:** `train_model(model, criterion, optimizer, epochs=10)`
   - Parameters: model, loss function, optimizer, number of training epochs
   - Return value: `train_losses, train_accs, test_accs` (lists of training loss, training accuracy, test accuracy)

**2. Internal initialization:**
- Create empty lists `train_losses`, `train_accs`, `test_accs` to record metrics for each epoch

**3. Training loop:**
- Outer loop: `for epoch in range(epochs)`
- Initialize each epoch: `epoch_loss = 0`, `correct = 0`, `total = 0`
- Call `model.train()` to set training mode
- Iterate through `trainloader`:
  - Unpack data: `images, labels = data`
  - Move data to device: `images, labels = images.to(device), labels.to(device)`
  - Forward propagation: `outputs = model(images)`
  - Compute loss: `loss = criterion(outputs, labels)`
  - Backward propagation: `loss.backward()`
  - Update parameters: `optimizer.step()`
  - Clear gradients: `optimizer.zero_grad()`
  - Compute batch correct predictions: `pred = outputs.argmax(dim=1)`, `correct += (pred == labels).float().sum().item()`
  - Accumulate `total += labels.size(0)`, `epoch_loss += loss.item()`

**4. Validation at the end of each epoch:**
- Call `model.eval()` to enter evaluation mode
- Use `with torch.no_grad():` to disable gradient computation
- Iterate through `testloader` to compute test accuracy
- Print format: `f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}"`

**5. Return the recorded metric lists**


In [ ]:
# TODO: add your code here

### 5.7 Execute Training


**Prompt**

Call the training function to train the model with the following requirements:
1. Call `train_model(model, criterion, optimizer, epochs=num_epochs)` to start training
2. Assign the return values to `train_losses`, `train_accs`, `test_accs` respectively


**Note:** The training process takes approximately 5-15 minutes (depending on CPU/GPU), please be patient.
Observe the changes in training loss, training accuracy, and test accuracy for each epoch.
Compared with the fully-connected network assignment — CNN test accuracy is expected to reach over 70%, far exceeding the fully-connected network's approximately 50%.


In [ ]:
# TODO: add your code here

### 5.8 Training Process Visualization


**Prompt**

Plot the loss and accuracy curves during training with the following requirements:
1. Create a canvas with `figsize=(12, 5)`
2. Left subplot `plt.subplot(1, 2, 1)`:
   - Plot `train_losses` curve
   - Title: `'Training Loss'`
   - X-axis label: `'Epoch'`, Y-axis label: `'Loss'`
3. Right subplot `plt.subplot(1, 2, 2)`:
   - Plot `train_accs` and `test_accs` curves
   - Add legend: `plt.legend(['Training Accuracy', 'Test Accuracy'])`
   - Title: `'Accuracy'`
   - X-axis label: `'Epoch'`, Y-axis label: `'Accuracy'`
4. Use `plt.tight_layout()` to adjust layout


**Think:**
- Observe the gap between training accuracy and test accuracy — is there overfitting?
- What is the approximate CNN test accuracy? Compared with the fully-connected network (about 50%), how large is the improvement?
- Why does CNN perform better than fully-connected networks on image tasks?


In [ ]:
# Plot training curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Training Accuracy')
plt.plot(test_accs, label='Test Accuracy')
plt.legend()
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.tight_layout()
plt.show()

print(f"Final test accuracy: {test_accs[-1]:.4f} ({test_accs[-1]*100:.2f}%)")


### 5.9 Model Evaluation (Confusion Matrix and Classification Report)


**Prompt**

Comprehensively evaluate the classification model performance on the test set using PyTorch with the following requirements:

**1. Evaluation process:**
- Call `model.eval()` to enter evaluation mode
- Create two empty lists `all_labels` and `all_preds` to store all true labels and predicted labels
- Use `with torch.no_grad():` to disable gradient computation
- Iterate through `testloader` to get data for each batch (`images, labels = data`), move to device
- Forward propagation: `outputs = model(images)`; get predicted classes: `pred = outputs.argmax(dim=1)`
- Extend `labels.cpu().numpy()` and `pred.cpu().numpy()` to the lists respectively

**2. Confusion matrix visualization:**
- Use `sklearn.metrics.confusion_matrix(all_labels, all_preds)` to compute confusion matrix
- Use `seaborn.heatmap` to plot heatmap:
  - Canvas size: `figsize=(10, 8)`
  - Parameters: `annot=True` (show values), `fmt='d'` (integer format), `cmap='Blues'`
  - X-axis tick labels: `classes`
  - Y-axis tick labels: `classes`
  - Title: `'CIFAR10 Confusion Matrix (CNN)'`
  - X-axis label: `'Predicted Class'`, Y-axis label: `'True Class'`

**3. Classification report:**
- Use `classification_report(all_labels, all_preds, target_names=classes)` to generate classification report
- Print the report

**4. Overall accuracy:**
- Use `accuracy_score(all_labels, all_preds)` to compute overall accuracy and print it


In [ ]:
# TODO: add your code here

## 6. Visualizing Convolution Kernels and Feature Maps

The first-layer convolution kernels of a CNN directly interact with the raw image. By visualizing these kernels and the feature maps they produce, we can understand what features the network has learned.


### 6.1 First-Layer Convolution Kernel Visualization


**Prompt**

Visualize the 32 convolution kernels learned by the first layer of the CNN with the following requirements:

**1. Get first-layer convolution kernel weights:**
- Get conv1 weights from model: `weights = model.conv1.weight.cpu().detach().numpy()`
- The weight shape is (32, 3, 3, 3), representing 32 kernels, each with 3 input channels, 3x3 size
- Take average across 3 channels: `kernels = weights.mean(axis=1)`, resulting in shape (32, 3, 3)

**2. Visualize all 32 kernels:**
- Create a 4x8 subplot canvas (`figsize=(16, 8)`), displaying 32 kernels
- Each subplot displays one kernel using `imshow(..., cmap='gray')`
- Title for each subplot: `f'Kernel {i+1}'`
- Turn off axes for all subplots
- Use `plt.suptitle('First-Layer Convolution Kernels (Averaged over 3 Channels)', fontsize=16)` to add main title
- Use `plt.tight_layout()` to adjust layout


**Think:**
- What similarities do these learned kernels have with previously hand-designed kernels (mean filter, Gaussian filter, Sobel edge detection)?
- Can you identify which kernels approximate edge detectors and which approximate blur filters?
- What does this demonstrate about CNNs?


In [ ]:
# TODO: add your code here

### 6.2 Feature Map Visualization


**Prompt**

Visualize the feature maps (feature maps) output by the first convolutional layer of the CNN with the following requirements:

**1. Extract feature maps:**
- Use the first image from the first batch in testloader as input: `for data in testloader: images, labels = data; break`
- Take the first image: `sample = images[0:1].to(device)` (keep batch dimension)

**2. Get first-layer feature maps:**
- Use `with torch.no_grad():` to disable gradient computation
- Pass through first conv layer: `features = model.conv1(sample)`
- Pass through BN and ReLU: `features = model.relu(model.bn1(features))`
- The feature map shape is (1, 32, 32, 32), take the first sample: `features = features[0].cpu().numpy()`

**3. Visualize first 16 feature maps:**
- Create a 4x4 subplot canvas (`figsize=(12, 12)`)
- Each subplot displays one feature map
- Use `cmap='viridis'` (or `gray`)
- Turn off axes
- Add main title `plt.suptitle('First-Layer Feature Maps (First 16 Channels)', fontsize=16)`
- Use `plt.tight_layout()`


**Think:**
- Which regions of the image do different channel feature maps focus on?
- What do bright and dark areas in the feature maps represent?
- What is the connection with the Sobel edge detection results in Section 4?
- Can you understand why CNNs are more suitable for image processing than fully-connected networks?


In [ ]:
# TODO: add your code here

## 7. Visualizing Pooling Operations

Pooling is an important operation in CNNs used to reduce the spatial dimensions of feature maps, decrease the number of parameters, and enhance translation invariance of features.


**Prompt**

Visualize the effects of Max Pooling and Average Pooling with the following requirements:

**1. Use a CIFAR10 image and convert to grayscale:**
- Use the first CIFAR10 image from testloader
- Convert to grayscale by averaging RGB channels

**2. Manual implementation of max pooling and average pooling:**
- Define `max_pool2d_manual(x, pool_size=2)` and `avg_pool2d_manual(x, pool_size=2)` functions
- Use `skimage.measure.block_reduce` with `np.max` and `np.mean` respectively
- Input is a 2D numpy array, output is pooled result

**3. Display comparison (`figsize=(12, 4)`):**
- 1 row, 3 columns:
  - (1,1): original grayscale image, title 'Original Grayscale (32x32)'
  - (1,2): max pooling result, title 'Max Pooling (16x16)'
  - (1,3): average pooling result, title 'Average Pooling (16x16)'
- All use `cmap='gray'`, `plt.axis('off')`
- Use `plt.tight_layout()`


In [ ]:
# TODO: add your code here

## 8. Data Augmentation

Data Augmentation improves model generalization and reduces overfitting by applying random transformations to training data.
This section will observe the effects of different data augmentation methods and compare the impact of using or not using data augmentation on CNN training.


### 8.1 Data Augmentation Methods Visualization


**Prompt**

Visualize the effects of different data augmentation methods in PyTorch with the following requirements:

**1. Use an image from the test set as the original image:**
- Use `images[0]` obtained previously (a torch Tensor)
- Convert this tensor back to a PIL Image using `transforms.ToPILImage()`

**2. Define different augmentation transforms:**
- `flip`: `transforms.RandomHorizontalFlip(p=1.0)` (always flip)
- `crop`: `transforms.RandomCrop(32, padding=4)`
- `jitter`: `transforms.ColorJitter(brightness=0.3, contrast=0.3)`
- `rotate`: `transforms.RandomRotation(degrees=15)`

**3. Display comparison (`figsize=(15, 3)`):**
- 1 row, 5 columns:
  - (1,1): original image, title 'Original'
  - (1,2): horizontal flip, title 'Horizontal Flip'
  - (1,3): random crop, title 'Random Crop'
  - (1,4): color jitter, title 'Color Jitter'
  - (1,5): rotation, title 'Rotation'
- All images need to be converted back to numpy array for display
- Turn off axes, use `plt.tight_layout()`


**Think:** What real-world changes do each of these augmentation methods simulate? How do they help model generalization?


In [ ]:
# TODO: add your code here

### 8.2 Comparison Experiment: Training With and Without Data Augmentation


**Prompt**

To verify the effect of data augmentation, create a comparison experiment without data augmentation with the following requirements:

**1. Create a training set loader without data augmentation:**
- Define `transform_no_aug = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])])`
- Create `trainset_no_aug = CIFAR10(root='./data', train=True, transform=transform_no_aug, download=True)`
- Create `trainloader_no_aug = DataLoader(trainset_no_aug, batch_size=batch_size, shuffle=True, num_workers=2)`

**2. Create a new model instance:**
- `model_no_aug = SimpleCNN().to(device)`
- `optimizer_no_aug = optim.Adam(model_no_aug.parameters(), lr=0.001)`

**3. Train for 5 epochs (quick comparison):**
- Modify the training function to accept a custom data loader parameter
- Or directly rewrite a simplified training loop
- Record `losses_aug` (with augmentation) and `losses_no_aug` (without augmentation)
- Record `test_acc_aug` and `test_acc_no_aug`

**4. Display comparison (`figsize=(12, 5)`):**
- Left: training loss comparison curve, legend ['With Augmentation', 'Without Augmentation']
- Right: test accuracy comparison curve, same legend
- Add appropriate titles and labels


**Think:**
- With data augmentation, does training loss decrease slower (because images seen each epoch are different)?
- Is test accuracy higher (because generalization is stronger)?
- Is the gap between training accuracy and test accuracy (overfitting degree) smaller?


In [ ]:
# TODO: add your code here

---

## 9. Reflection and Summary

After completing the above experiments, please answer the following questions:

### Part 1: Convolution Operation Fundamentals

1. **Mathematical meaning of convolution:** Please explain the definition of image convolution operation and why boundary padding (padding) is necessary?

2. **Kernel design:** Please explain the design principles and effects of mean filter, Gaussian filter, and sharpening kernels.

3. **Noise and denoising:** Why is median filtering more effective than Gaussian filtering for salt & pepper noise?

### Part 2: Edge Detection

4. **Sobel operator:** How does the Sobel operator detect edges? What are the advantages of Gaussian first-order derivatives over Sobel?

5. **Connection with CNN:** What is the connection between the first-layer convolution kernels learned by CNN and hand-designed edge detection operators?

### Part 3: CNN Architecture

6. **Why CNN outperforms FC:** Why can CNN achieve higher accuracy than fully-connected networks on image tasks? (Hint: consider local connectivity, weight sharing, translation invariance)

7. **Role of pooling:** What is the function of pooling layers? Why is max pooling more commonly used than average pooling?

8. **Data augmentation:** Why can data augmentation reduce overfitting? Which types of image tasks benefit most from data augmentation?

### Part 4: Comprehensive Thinking

9. **Comparison of manual kernels and learned kernels:** What are the similarities and differences between manually designed convolution kernels (Sobel, Gaussian) and kernels learned by CNN? What are their respective advantages?

10. **Prompt programming reflection:** When using AI programming tools, how can you write effective prompts? Please summarize your experience and tips.
